In [1]:
import pickle
from dgllife.model.model_zoo import WeavePredictor
import dgl
import dgl.nn as nn
import dgl.function as fn
import torch.nn as tnn
import torch
import torch.optim
import torch.nn.functional as F
from torch.utils.data import random_split
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from pprint import pprint
from sklearn.preprocessing import MinMaxScaler

In [2]:
with open('graph_ndatas.pickle', 'rb') as handle:
    ndatas = pickle.load(handle)

with open('graph_edatas.pickle', 'rb') as handle:
    edatas = pickle.load(handle)
    
with open('graphs.pickle', 'rb') as handle:
    graphs = pickle.load(handle)

In [3]:
def OHE_and_normalize(mol: str):
    n = []
    for i in range(graphs[mol].num_nodes()):
        for x in ndatas[mol]:
            if (x[0] == i):
                n.append(x[1:7]+[x[-1][0]])
                break
            elif (x[7] == i):
                n.append(x[8:14]+[x[-1][1]])
                break
    assert len(n) == graphs[mol].num_nodes()
    
    e = []
    for i, j in zip(graphs[mol].edges()[0].tolist(), graphs[mol].edges()[1].tolist()):
        for x in ndatas[mol]:
            if ((i, j) == (x[0], x[7])) or ((i, j) == (x[7], x[0])):
                e.append(x[-3:-1])
                break
        for y in edatas[mol].keys():
            if ((str(i), str(j)) == y) or ((str(j), str(i)) == y):
                e[-1].append(edatas[mol][y])
    assert len(e) == graphs[mol].num_edges()
            
    #n_df = pd.DataFrame(n)
    #encoded_column_1 = pd.get_dummies(n_df[0])
    #encoded_column_2 = pd.get_dummies(n_df[6], prefix="nei")
    #n_df = n_df.join(encoded_column_1)
    #n_df = n_df.join(encoded_column_2)
    #print(n_df)
    #df.drop(col, axis=1, inplace=True)
    # try:
    #df = df.join(encoded_column)
    # except ValueError as verr:
    #     if "overlap" in verr.__str__():
    #         df = df.merge(encoded_column, left_on="BR", right_on="BR")
    return n, e

In [4]:
sorted_keys = sorted([int(x) for x in ndatas.keys()])
n_all = []
e_all = []
for x in sorted_keys:
        x_n, x_e = OHE_and_normalize(str(x))
        n_all = n_all+x_n
        e_all = e_all+x_e


In [5]:
n_df = pd.DataFrame(n_all, columns=["element", "atomic_number", "radius", "mass", "electronegativity", "hybridisation", "nei"])
e_df = pd.DataFrame(e_all)
ele_encoded = pd.get_dummies(n_df["element"], prefix="ele_")
nei_encoded = pd.get_dummies(n_df["nei"], prefix="nei")
n_df.drop("element", axis=1, inplace=True)
n_df.drop("nei", axis=1, inplace=True)
n_df = n_df.join(ele_encoded)
n_df = n_df.join(nei_encoded)
norm = MinMaxScaler().fit(n_df)
norm_n_df = norm.transform(n_df)

In [6]:
comb_graph = dgl.batch([graphs[str(x)] for x in sorted_keys])
#nx.draw(dgl.to_networkx(graphs['21']), with_labels=True)

In [7]:
class GraphConv(tnn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = nn.SAGEConv(in_feats=in_feats, out_feats=hid_feats, bias=True, aggregator_type='mean')
        self.conv3 = nn.SAGEConv(in_feats=hid_feats, out_feats=out_feats, bias=True, aggregator_type='mean')
        
    def message_passing(self, g):

        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
    
    def forward(self, graph, inputs):
        self.message_passing(graph)
        
        h = self.conv1(graph, inputs)
        h = F.relu(h)
        h = self.conv3(graph, h)
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(fn.u_dot_v('h', 'h', 'score'))
            return graph.edata['score']
        

In [8]:
e_star = e_df.drop(2, axis=1)
norm_e = MinMaxScaler().fit(e_star)
norm_e_star = norm_e.transform(e_star)

In [9]:
comb_graph.ndata['h'] = torch.from_numpy(norm_n_df.astype('float32'))
comb_graph.edata['e'] = torch.from_numpy(norm_e_star.astype('float32'))
comb_graph.edata['y'] = torch.from_numpy(e_df[2].to_numpy().astype('float32'))

In [16]:
train_sp, val_sp, test_sp = random_split(norm_e_star, [0.5, 0.25, 0.25])
train_bin_full, train_bin, val_bin, test_bin = (np.zeros(len(comb_graph.edata['e'])) for i in range(4))
for x in train_sp.indices:
    train_bin[x] = 1
for y in val_sp.indices:
    val_bin[y] = 1
for z in test_sp.indices:
    test_bin[z] =1

for x in range(len(train_bin_full)):
    train_bin_full[x] = 1
print(train_bin_full)
print(train_bin)

[1. 1. 1. ... 1. 1. 1.]
[0. 1. 1. ... 1. 1. 1.]


In [17]:
comb_graph.edata['train_mask'] = torch.from_numpy(train_bin_full).bool()
comb_graph.edata['val_mask'] = torch.from_numpy(val_bin).bool()
comb_graph.edata['test_mask'] = torch.from_numpy(test_bin).bool()

In [18]:
node_features = comb_graph.ndata['h']
edge_label = comb_graph.edata['y']
train_mask = comb_graph.edata['train_mask']


In [19]:
self_loop_g = dgl.add_self_loop(comb_graph)

In [20]:
model = GraphConv(self_loop_g.ndata['h'].shape[1], 20, 10)
optimizer = torch.optim.Adam(model.parameters())


In [ ]:
optimizer.param_groups[0]['lr'] = 0.001

for epoch in range(700):
        pred = model(comb_graph, node_features)
        loss = abs(pred[train_mask] - edge_label[train_mask]).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())

In [22]:
model.load_state_dict(torch.load('model_parms.pt'))

<All keys matched successfully>

In [48]:
(edge_label[train_mask])

tensor([180040.1094, 298029.2500, 268233.4688,  ..., 185079.9219,
        198301.7969, 123835.5703])

In [49]:
(pred[train_mask])


tensor([[259536.7500],
        [252575.1719],
        [252575.1719],
        ...,
        [252028.8906],
        [262564.3438],
        [251043.8281]], grad_fn=<IndexBackward0>)

In [64]:
abs(pred[train_mask] - edge_label[train_mask]).mean()

tensor(81162.5938, grad_fn=<MeanBackward0>)

In [24]:
torch.save(model.state_dict(), 'model_parms_80k.pt')

In [ ]:
model = GraphConv(comb_graph.ndata['h'].shape[1], 50, 20, 10)
optimizer = torch.optim.Adam(model.parameters())

model_weave = WeavePredictor(comb_graph.ndata['h'].shape[1], comb_graph.edata['e'].shape[1], 

In [42]:
print(norm_e_star)

[[0.37620873 0.20721817]
 [0.08499796 0.18207624]
 [0.0850565  0.18207624]
 ...
 [0.00174659 0.10583942]
 [0.08424243 0.17964315]
 [0.08451123 0.17680454]]
